# LoRA Training Notebook

Thin diagnostic wrapper over the canonical training runtime in `experiments.training.train_adapter.pipeline`.

## Sections
1. Setup
2. Compose training config
3. Preview resolved config
4. Run canonical training pipeline
5. Inspect run outputs
6. Inspect MLflow run metadata

## 1. Import Required Libraries

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

# Match the Python import layout used by the CLI and Airflow DAG.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path("../..").resolve())).resolve()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

mlflow = importlib.import_module("mlflow")

from experiments.training.train_adapter.config import register_configs

register_configs()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print("All imports OK.")

## 2. Compose Training Config

Load the same Hydra config the CLI and DAG use, then apply lightweight overrides
for a short notebook run. The notebook does not rebuild the trainer or model
manually; it just prepares config for the canonical pipeline.

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from experiments.training.train_adapter.config import load_app_config

ACCELERATOR = "auto"  # "auto" | "gpu" | "cpu"
MAX_EPOCHS = 1
BATCH_SIZE = 1
NUM_WORKERS = 0
LIMIT_TRAIN_BATCHES = 5
LIMIT_VAL_BATCHES = 2

HYDRA_OVERRIDES = [
    f"experiment.trainer.accelerator={ACCELERATOR}",
    f"experiment.trainer.max_epochs={MAX_EPOCHS}",
    f"experiment.trainer.limit_train_batches={LIMIT_TRAIN_BATCHES}",
    f"experiment.trainer.limit_val_batches={LIMIT_VAL_BATCHES}",
    "experiment.trainer.val_check_interval=1.0",
    "experiment.trainer.log_every_n_steps=1",
    f"experiment.data.batch_size={BATCH_SIZE}",
    f"experiment.data.num_workers={NUM_WORKERS}",
]

CONF_DIR = str(PROJECT_ROOT / "experiments" / "training" / "conf")

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=CONF_DIR, version_base=None):
    raw_cfg = compose(config_name="config", overrides=HYDRA_OVERRIDES)

app_cfg = load_app_config(raw_cfg)

print("Hydra overrides:")
for item in HYDRA_OVERRIDES:
    print(f"  {item}")

print("\nResolved high-level settings:")
print(f"  project_root : {app_cfg.paths.project_root}")
print(f"  model_path   : {app_cfg.experiment.model.local_path}")
print(f"  batch_size   : {app_cfg.experiment.data.batch_size}")
print(
    f"  trainer      : {app_cfg.experiment.trainer.accelerator} / {app_cfg.experiment.trainer.precision}"
)

## 3. Preview Resolved Config

This is the exact Hydra config that will be passed into the canonical
`run_training()` function.

In [ ]:
from omegaconf import OmegaConf

print(OmegaConf.to_yaml(raw_cfg, resolve=True))

## 4. Run Canonical Training Pipeline

This cell calls `experiments.training.train_adapter.pipeline.run_training()`
directly. No manual trainer, callback, logger, datamodule, or model wiring is
reconstructed in the notebook.

In [ ]:
import traceback

from experiments.training.train_adapter.pipeline import run_training

run_id = None
save_dir = None
run_dir = None

try:
    run_id, save_dir, run_dir = run_training(raw_cfg)
    print("Training finished successfully.")
    print(f"  run_id   : {run_id}")
    print(f"  export   : {save_dir}")
    print(f"  run_dir  : {run_dir}")
except Exception:
    print("\n" + "=" * 60)
    print("TRAINING FAILED — full traceback:")
    print("=" * 60)
    traceback.print_exc()
    print("=" * 60)

## 5. Inspect Run Outputs

List the files written by the canonical runtime under the run-scoped artifact
directory.

In [ ]:
if not run_dir:
    print("No completed run yet.")
else:
    run_path = Path(run_dir)
    print(f"Run directory : {run_path}")

    for subdir in ["checkpoints", "export", "metadata", "evaluation"]:
        target = run_path / subdir
        print(f"\n{subdir}/")
        if not target.exists():
            print("  <missing>")
            continue

        for child in sorted(target.rglob("*")):
            rel = child.relative_to(run_path)
            suffix = "/" if child.is_dir() else ""
            print(f"  {rel}{suffix}")

## 6. Inspect MLflow Run

If the run completed and MLflow is configured, fetch the recorded run metadata
for quick inspection.

In [ ]:
if not run_id:
    print("No run_id available yet.")
else:
    tracking_uri = os.getenv("MLFLOW_BACKEND_URI")
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    run = mlflow.get_run(run_id)
    print(f"Tracking URI : {mlflow.get_tracking_uri()}")
    print(f"Run name     : {run.data.tags.get('mlflow.runName')}")
    print(f"Status       : {run.info.status}")

    if run.data.metrics:
        print("\nMetrics:")
        for key, value in sorted(run.data.metrics.items()):
            print(f"  {key:<24} {value}")
    else:
        print("\nNo metrics recorded on the run.")

## Notes

This notebook is intentionally thin. It exists to compose overrides, call the
canonical runtime, and inspect the resulting artifacts and run metadata.

In [ ]:
print("This notebook does not rebuild the training runtime by hand.")
print(
    "It delegates the actual training run to experiments.training.train_adapter.pipeline.run_training()."
)
print("Use experiments/training/lora_ops.ipynb for registration, promotion, and sync operations.")